# Grasp adaptation — compliance-matched stiffness control

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
from pathlib import Path

sys.path.insert(0, os.path.join('../'))
from hand_config import K_TIP_GENTLE, K_TIP_PROBE, K_TIP_HOLD, K_GAIN, K_MIN, K_MAX

OUTPUT_DIR = os.path.join('outputs', 'grasp_adaptation')
OBJECTS    = ['hard_obj', 'soft_obj']
OBJ_LABELS = {'hard_obj': 'Hard', 'soft_obj': 'Soft'}
COLORS     = {'hard_obj': '#0072B2', 'soft_obj': '#D55E00'}

def load_latest(obj):
    folder = Path(OUTPUT_DIR)
    if not folder.exists():
        return None
    files = sorted(folder.glob(f'grasp_{obj}*.csv'))
    return pd.read_csv(files[-1]) if files else None

data = {obj: load_latest(obj) for obj in OBJECTS}
print('Loaded:', {obj: (len(data[obj]) if data[obj] is not None else 'no data') for obj in OBJECTS})

## Thumb compliance estimate $C_O^{\rm thumb}$

$C_O = \|\Delta x_{\rm thumb}\| / \|\Delta F_{\rm thumb}\|$ from the sense → probe finite difference. Lower = stiffer object.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for ax, obj in zip(axes, OBJECTS):
    df = data[obj]
    if df is None:
        ax.set_title(OBJ_LABELS[obj] + ' — no data')
        continue

    sense = df[df['phase'] == 'sense']
    probe = df[df['phase'] == 'probe']
    if sense.empty or probe.empty:
        ax.set_title(OBJ_LABELS[obj] + ' — missing phases')
        continue

    pos_s = sense[['tip_thumb_x_m','tip_thumb_y_m','tip_thumb_z_m']].mean().to_numpy()
    F_s   = sense[['force_1st_thumb_x_N','force_1st_thumb_y_N','force_1st_thumb_z_N']].mean().to_numpy()
    pos_p = probe[['tip_thumb_x_m','tip_thumb_y_m','tip_thumb_z_m']].mean().to_numpy()
    F_p   = probe[['force_1st_thumb_x_N','force_1st_thumb_y_N','force_1st_thumb_z_N']].mean().to_numpy()

    d_pos = pos_p - pos_s
    d_F   = F_p   - F_s
    nF    = np.linalg.norm(d_F)
    C_O   = np.linalg.norm(d_pos) / nF if nF > 1e-12 else np.nan
    k_applied = float(probe['k_applied_Npm'].median())

    color = COLORS[obj]
    ax.bar(['$C_O$'], [C_O * 1e3], color=color, alpha=0.85, width=0.4)
    ax.set_ylabel(r'$C_O^{\mathrm{thumb}}$ [mm/N]')
    ax.set_title(OBJ_LABELS[obj], fontsize=13, fontweight='bold')
    ax.text(0, C_O * 1e3 * 1.05,
            f'{C_O*1e3:.1f} mm/N\n→ $k_{{\rm app}}$={k_applied:.0f} N/m',
            ha='center', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, None)

fig.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_compliance.pdf'), bbox_inches='tight')
plt.show()

## Stiffness timeline

$k_{tip}$ over time for each object, showing the sense → probe → adapt sequence.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for obj in OBJECTS:
    df = data[obj]
    if df is None:
        continue
    t0 = df['time_s'].iloc[0]
    ax.plot(df['time_s'] - t0, df['k_tip_Npm'],
            color=COLORS[obj], lw=2, label=OBJ_LABELS[obj])
    # shade phases
    for phase, alpha in [('sense', 0.10), ('probe', 0.15), ('adapt', 0.10), ('hold', 0.12)]:
        sub = df[df['phase'] == phase]
        if sub.empty: continue
        ax.axvspan(sub['time_s'].iloc[0] - t0, sub['time_s'].iloc[-1] - t0,
                   alpha=alpha, color=COLORS[obj])

ax.set_xlabel('Time [s]')
ax.set_ylabel(r'$k_{tip}$ [N/m]')
ax.axhline(K_TIP_GENTLE, color='0.5', lw=1, linestyle=':', label=r'$k_{\rm gentle}$')
ax.axhline(K_TIP_PROBE,  color='0.5', lw=1, linestyle='--', label=r'$k_{\rm probe}$')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'stiffness_timeline.pdf'), bbox_inches='tight')
plt.show()

## Thumb force during hold

$\|F_{\rm thumb}\|$ over time in the hold phase (adapted stiffness active).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for obj in OBJECTS:
    df = data[obj]
    if df is None:
        continue
    hold = df[df['phase'] == 'hold']
    if hold.empty:
        continue
    t0   = hold['time_s'].iloc[0]
    Fmag = np.linalg.norm(
        hold[['force_1st_thumb_x_N','force_1st_thumb_y_N','force_1st_thumb_z_N']].to_numpy(),
        axis=1)
    k_app = float(hold['k_applied_Npm'].median())
    ax.plot(hold['time_s'] - t0, Fmag,
            color=COLORS[obj], lw=2,
            label=f'{OBJ_LABELS[obj]} ($k_{{\rm app}}$={k_app:.0f} N/m)')

ax.set_xlabel('Time in hold [s]')
ax.set_ylabel(r'$\|F_{\rm thumb}\|$ [N]')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_force_hold.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import sys
sys.path.insert(0, os.path.join('../'))
from ModelIDHand.hand_viz import plot_hand, _COL

FINGERTIPS_VIZ = ['thumb', 'index', 'middle', 'ring', 'pinky']
_Q_COLS = [f'q_motor_{i}_rad' for i in range(13)]
_STYLE  = dict(linestyle='-', linewidth=4.5, marker='o', markersize=6.0)
_ELEV, _AZIM = 22, 112

objects_with_data = [o for o in COLORS if data[o] is not None]
fig = plt.figure(figsize=(7 * len(objects_with_data), 6))
fig.patch.set_facecolor('white')
axes = []
for i, obj in enumerate(objects_with_data):
    df = data[obj]
    hold = df[df['phase'] == 'hold']
    q    = hold[_Q_COLS].iloc[len(hold) // 2].to_numpy(dtype=np.float64)
    ax3d = fig.add_subplot(1, len(objects_with_data), i + 1, projection='3d')
    plot_hand(q, ax=ax3d, style=_STYLE)
    leg = ax3d.get_legend()
    if leg: leg.remove()
    ax3d.set_xticks([]); ax3d.set_yticks([]); ax3d.set_zticks([])
    for pane in (ax3d.xaxis.pane, ax3d.yaxis.pane, ax3d.zaxis.pane):
        pane.fill = False; pane.set_edgecolor('#d8d8d8')
    ax3d.view_init(elev=_ELEV, azim=_AZIM)
    post = df[df['phase'].isin(['adapt', 'lift', 'hold', 'place'])]
    k_app = float(post['k_applied_Npm'].iloc[0]) if not post.empty else float('nan')
    ax3d.set_title(f"{obj.replace('_', ' ').title()}\n$k_{{applied}}={k_app:.0f}$ N/m",
                   fontsize=14, fontweight='semibold', pad=14)
    axes.append(ax3d)

# sync axes limits
lims = np.array([[a.get_xlim3d(), a.get_ylim3d(), a.get_zlim3d()] for a in axes])
xl = [lims[:, 0, 0].min(), lims[:, 0, 1].max()]
yl = [lims[:, 1, 0].min(), lims[:, 1, 1].max()]
zl = [lims[:, 2, 0].min(), lims[:, 2, 1].max()]
span = max(xl[1]-xl[0], yl[1]-yl[0], zl[1]-zl[0])
cx, cy, cz = sum(xl)/2, sum(yl)/2, sum(zl)/2
for a in axes:
    a.set_xlim3d(cx-span/2, cx+span/2)
    a.set_ylim3d(cy-span/2, cy+span/2)
    a.set_zlim3d(cz-span/2, cz+span/2)

_LEG = [plt.Line2D([0],[0], color=_COL[n], lw=5, label=n.capitalize())
        for n in FINGERTIPS_VIZ]
fig.legend(handles=_LEG, loc='lower center', ncol=5, fontsize=10,
           frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'hold_hand_pose.pdf'), bbox_inches='tight', dpi=300)
plt.show()